# Kazakh Profanity Detection — fine-tuning XLM-RoBERTa (Colab)

This notebook runs the whole project in a **fresh Google Colab runtime**:
data preparation → ablation (raw / normalized / normalized + augmentation) → training → one evaluation on the test set →
predictions on your own text → saving the model.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Total time ≈ 40 minutes (three training runs).

If you only want to **use the trained model** (no training), skip to section 7.

In [ ]:
!nvidia-smi -L   # should show a GPU (e.g. Tesla T4)

## 1. Get the code and install packages

In [ ]:
# The GitHub repository must be PUBLIC (or upload the project zip to Colab and unzip it instead of cloning).
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/kazakh-profanity-detection.git"  # <- change
!git clone -q $REPO_URL
%cd kazakh-profanity-detection
!pip install -q -r requirements.txt

## 2. Prepare the dataset
The labeled splits are already in `data/processed/`. Running this cell rebuilds them from the raw
KazSAnDRA files + my manual labels + the classmate's labels (seed 42) and produces **identical** files.

In [ ]:
!python src/prepare_data.py
import json, pandas as pd
pd.set_option("display.max_colwidth", 120)
info = json.load(open("data/processed/split_info.json"))
pd.DataFrame({k: {m: v[m] for m in ["total", "profane_1", "clean_0", "synthetic"]} for k, v in info["splits"].items()}).T

In [ ]:
train = pd.read_csv("data/processed/train.csv", keep_default_na=False)
print(train[train.label == 1].profanity_type.value_counts(), "\n")
train.groupby("label").sample(6, random_state=1)[["text", "label", "profanity_type", "script", "source"]]

### Text normalization (what the model actually sees)
Latin → Kazakh Cyrillic, Latin look-alike letters fixed, long repeats shortened. Kazakh letters are kept.

In [ ]:
import sys; sys.path.insert(0, "src")
from normalize import normalize_text
for t in ["Qotaqbas oyin", "kotak", "shesheńdi mynaý", "Mal eken", "бляяяяя", "Free Fire керемет"]:
    print(f"{t:25} -> {normalize_text(t)}")

## 3. Training + ablation (with / without each preprocessing step)
Settings are in `config.json` (lr 2e-5, batch 16, 4 epochs, max 128 tokens, seed 42, **class-weighted loss**).

**Ablation (lecture: training ablation):** the same model is trained 3 times, switching parts of the pipeline on:

| run | normalization | augmentation |
|---|---|---|
| raw text | off | off |
| normalized | on | off |
| full (main) | on | on |

Plus a **baseline without ML** (word-list rules). All are compared on the **validation** set; the test set is
used only once, for the chosen model. Each run takes ~10–12 min on a T4.

In [ ]:
!python src/train.py --normalize 0 --augment 0 --output_dir model/xlmr-raw --results_dir results/ablation_raw

In [ ]:
!python src/train.py --normalize 1 --augment 0 --output_dir model/xlmr-norm --results_dir results/ablation_norm

In [ ]:
!python src/train.py   # full pipeline -> model/xlmr-kaz-profanity

In [ ]:
!python src/compare_ablation.py
import shutil
chosen = json.load(open("results/ablation.json"))["chosen"]
other = {"XLM-R, raw text": "model/xlmr-raw", "XLM-R, normalized text": "model/xlmr-norm"}
if chosen in other:   # keep the chosen model under the standard name
    shutil.move("model/xlmr-kaz-profanity", "model/xlmr-full")
    shutil.move(other[chosen], "model/xlmr-kaz-profanity")
print("Final model:", chosen)

## 4. Evaluate ONCE on the held-out test set (only the chosen model)

In [ ]:
!python src/evaluate.py
from IPython.display import Image, display
display(Image("results/confusion_matrix.png"))
m = json.load(open("results/test_metrics.json"))
display(pd.DataFrame(m["recall_by_profanity_type"]).T)
display(pd.DataFrame(m["recall_by_script"]).T)
display(pd.DataFrame(m["false_positive_rate_by_source"]).T)
print("Word-list baseline on the same test set:", m["lexicon_baseline"])
print("95% bootstrap confidence intervals:", m["bootstrap_95ci"])
print("Calibration (ECE, temperature):", m["calibration"])
print("Abstention (unsure reviews -> human):", m["abstention_0.2_0.8"])
print("\nRobustness: the same test reviews rewritten (masked / symbols / Latin / no Kazakh letters)")
display(pd.DataFrame(json.load(open("results/robustness.json"))).T)

In [ ]:
fp = pd.read_csv("results/errors_false_positives.csv", keep_default_na=False)
fn = pd.read_csv("results/errors_false_negatives.csv", keep_default_na=False)
print("False positives (clean, but predicted profane):", len(fp))
display(fp[["text", "probability", "source"]].head(10))
print("False negatives (profane, but missed):", len(fn))
display(fn[["text", "probability", "profanity_type"]].head(10))

## 5. Try your own comments

In [ ]:
from predict import ProfanityDetector, show
det = ProfanityDetector("model/xlmr-kaz-profanity")
show(det.predict([
    "Тауар сапалы екен, жылдам жеткізді, рахмет!",
    "Өте нашар қызмет, ақшамды қайтарыңдар",
    "Мына қотақбастар ойынды бұзып тастады",
    "Qotaqbas oyin eken",
    "Мал екенсіңдер, ақшамды қайтарыңдар",
    "Мал базары жақсы, таза екен",
]))

In [ ]:
text = input("Enter a Kazakh comment: ")
show(det.predict(text))

### Trace one comment from raw text to prediction (useful for the defense)

In [ ]:
import torch
from utils import clean_text
text = "Мына qotaqbastar oyindy buzdy"
clean = clean_text(text, det.cfg["normalize"])                     # 1. whitespace + normalization
tokens = det.tokenizer.tokenize(clean)                             # 2. sub-word pieces
enc = det.tokenizer(clean, return_tensors="pt").to(det.device)     # 3. ids + attention mask (+ <s>, </s>)
print("normalized:", clean)
print("tokens:", tokens)
print("input_ids:", enc["input_ids"][0].tolist())
with torch.no_grad():
    logits = det.model(**enc).logits                               # 4. model -> 2 logits
probs = torch.softmax(logits, dim=-1)[0]                           # 5. softmax -> probabilities
thr = det.cfg["threshold"]
print("logits:", logits[0].tolist())
print(f"p(clean)={probs[0]:.4f}  p(profane)={probs[1]:.4f}  threshold={thr}")
print("prediction:", int(probs[1] >= thr))                         # 6. compare with threshold

## 6. Save the model and results
The model folder (~560 MB) is too big for GitHub, so we put it on Google Drive.
After this cell: open Google Drive → `xlmr-kaz-profanity.zip` → Share → **Anyone with the link** → copy the link
and paste it into the README (section "Model download").

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
!cd model && zip -r -q /content/drive/MyDrive/xlmr-kaz-profanity.zip xlmr-kaz-profanity
!ls -lh /content/drive/MyDrive/xlmr-kaz-profanity.zip
!zip -r -q results.zip results
files.download("results.zip")   # commit these result files to GitHub

## 7. Use the trained model without retraining (fresh runtime)
Run section 1 first (clone + install), then this cell. Put the Google Drive file ID
(the long code in the share link `https://drive.google.com/file/d/<FILE_ID>/view`) below.

In [ ]:
FILE_ID = "PASTE_FILE_ID_HERE"
!gdown $FILE_ID -O model.zip
!unzip -q -o model.zip -d model
import sys; sys.path.insert(0, "src")
from predict import ProfanityDetector, show
det = ProfanityDetector("model/xlmr-kaz-profanity")
show(det.predict(input("Enter a Kazakh comment: ")))